I am thinking about a few different options for the final project:

### CFD for city block/local weather conditions
- Build something similar to the urban weather modeler, that models/estimates the differences in temperature between buildings (e.g., the canyon effect, etc.)
- CFD is a highly complex process to analyze 


---
Urban heat islands and street-canyon effects create significant temperature gradients across just tens of meters. Tall buildings trap heat, block wind, and create recirculation zones that are invisible in regional weather models but felt by pedestrians. A numerical model can make these local effects visible.

### Physical Problem
We want to solve the **incompressible Navier-Stokes equations** coupled with a **scalar heat transport equation** over a 2-D cross-section of a city block:

$$\nabla \cdot \mathbf{u} = 0$$

$$\rho \left( \frac{\partial \mathbf{u}}{\partial t} + \mathbf{u} \cdot \nabla \mathbf{u} \right) = -\nabla p + \mu \nabla^2 \mathbf{u} + \rho \mathbf{g} \beta (T - T_\infty)$$

$$\frac{\partial T}{\partial t} + \mathbf{u} \cdot \nabla T = \alpha \nabla^2 T + Q_{solar}$$

The last term in the momentum equation is the **Boussinesq buoyancy approximation** — it lets temperature drive flow without fully solving compressible flow.

### Key Phenomena to Capture
| Effect | Physical Mechanism |
|---|---|
| Street canyon heating | Radiative trapping between building walls |
| Wind shadow / recirculation | Flow separation at building corners |
| Nocturnal cooling | Longwave emission from rooftops vs. trapped street air |
| Green space mitigation | Evapotranspiration as a local heat sink |

### Tire degradation/f1 models
- Build a physically consistent model of a f1 tire throughout a race - complex thermodyanmics system, with interesting forces exerted and unique edge cases


### Numerical Approach

#### Discretization strategy
- **Spatial**: Finite Difference (FD) or Finite Volume (FV) on a structured staggered grid (Arakawa C-grid). Staggering avoids pressure-velocity decoupling.
- **Pressure-velocity coupling**: Projection method (Chorin) — solve a provisional velocity field, then correct via a Poisson solve for pressure.
- **Time integration**: Semi-implicit Crank–Nicolson for diffusion (stability), explicit Adams-Bashforth for advection.
- **Turbulence**: Smagorinsky LES sub-grid model, or a simple mixing-length RANS model for a first pass.

#### Boundary Conditions
| Boundary | Velocity | Temperature |
|---|---|---|
| Inflow (left) | Log-law wind profile | Ambient $T_\infty$ |
| Outflow (right) | Zero-gradient / convective | Zero-gradient |
| Ground & walls | No-slip | Fixed $T_{wall}$ (solar load) |
| Top | Free-slip | Fixed $T_\infty$ |

#### Key Numerical Challenges
1. **Pressure Poisson equation** — dominant cost; use multigrid or FFT-based solver
2. **CFL condition** — time step constrained by `dt < dx/u_max`; adaptive stepping needed
3. **Sharp corners** on buildings cause flow separation — needs fine local resolution
```

### Expected Deliverables
- Steady-state velocity streamlines showing recirculation in street canyons
- Temperature contour maps comparing canyon vs. open scenarios
- Parametric study: building height ratio $H/W$ vs. peak street temperature

## Tire Degradation Models 

---
## Project 2 — F1 Tire Degradation Model

### Motivation
A Formula 1 tire is arguably the most complex consumable in motorsport. Its performance is governed by the coupled interaction of thermodynamics, viscoelastic rubber mechanics, and contact patch dynamics — all evolving over a 20–50 lap stint. Getting the model right determines pit-stop strategy.


### Governing Equations

**1. Thermal model** (lumped 3-layer: surface, bulk rubber, carcass)

$$m_s c_s \frac{dT_s}{dt} = \dot{Q}_{friction} - h_{conv}(T_s - T_{air}) - k_{sb}(T_s - T_b)$$

$$m_b c_b \frac{dT_b}{dt} = k_{sb}(T_s - T_b) - k_{bc}(T_b - T_c)$$

$$m_c c_c \frac{dT_c}{dt} = k_{bc}(T_b - T_c) - h_{rim}(T_c - T_{rim})$$

**2. Frictional heat generation**

$$\dot{Q}_{friction} = \mu(T_s, \delta) \cdot F_z \cdot v_{slip}$$

where $\delta$ is tread depth and $\mu(T_s)$ peaks near the compound's optimal window (~90–110 °C for soft compounds).

**3. Wear / degradation model**

$$\frac{d\delta}{dt} = -C_w \cdot \mu(T_s) \cdot F_z^\alpha \cdot v_{slip}^\beta$$

**4. Tire pressure**

$$p(t) = p_0 \frac{T_b(t)}{T_{b,0}}  \quad \text{(ideal gas, fixed volume)}$$

### Key Edge Cases & Nonlinearities

| Phenomenon | Numerical Challenge |
|---|---|
| Thermal cliff (graining) | Sharp bifurcation in $\mu(T)$ — stiff ODE |
| Blistering threshold | Discontinuous wear rate — event detection |
| Safety car / slow laps | Tire cools below window; restart transient |
| Wet track | $\mu$ drops dramatically; aquaplaning threshold |
| Undercut strategy | Multi-stint optimization across discrete compounds |

### Numerical Approach

#### ODE Solver Strategy
The system is a **stiff ODE** — the fast thermal surface dynamics (τ ~ seconds) are coupled to slow tread wear (τ ~ lap). This calls for implicit or semi-implicit methods:
- `scipy.integrate.solve_ivp` with `method='Radau'` or `'BDF'` — handles stiffness robustly
- Event detection via `events=` argument to catch blistering/graining thresholds

#### Extensions Worth Exploring
1. **Track-position-resolved model**: Replace scalar $F_z$, $v_{slip}$ with lap-by-lap telemetry inputs → ODE driven by a data signal
2. **Optimization / strategy layer**: Given the degradation ODE, solve for optimal pit-stop lap via dynamic programming or a simple 1-D search
3. **Monte Carlo uncertainty**: Treat $C_w$, $\mu_{peak}$ as random variables; propagate uncertainty across a race distance


### Expected Deliverables
- Temperature & wear trajectories for a full stint on each compound
- Phase portrait in $(T_s, \delta)$ space showing approach to the thermal cliff
- Strategy comparison: 1-stop vs. 2-stop, soft vs. medium compounds
- Sensitivity analysis: which parameters most affect lap-time loss per lap?